# 01 — Análisis exploratorio

> **El notebook explora y narra; no define lógica.** Todo lo que aquí resulte
> útil se muda a `src/miproyecto/` y vuelve importado. Un notebook que contiene
> la definición de las features es un notebook que nadie puede testear, y es
> exactamente lo que el proyecto pide evitar.
>
> Los outputs se eliminan al commitear (hook `nbstripout`): son ruido ilegible en
> el diff y filtran rutas y nombres de usuario de quien lo ejecutó.

**TODO(estudiante) 27:** completa las cinco secciones. El EDA es el entregable
del arranque del proyecto, no un adorno: de aquí salen el contrato de datos, la estrategia de
imputación y la justificación del split.

In [ ]:
from pathlib import Path

import pandas as pd

# Se importa del paquete: una sola definición de features en todo el proyecto.
from miproyecto import config
from miproyecto.data import contract as dc
from miproyecto.data import loaders
from miproyecto.features import contract as fc

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

print("particiones declaradas:", [p.etiqueta for p in config.TODAS_LAS_PARTICIONES])

## 1. Procedencia y contrato

Antes de mirar una sola distribución: de dónde viene el dato, con qué licencia,
cuándo se descargó y qué esquema se espera. Si esto no está claro, el resto del
análisis no es auditable.

In [ ]:
# Se usa el fixture versionado para que el notebook corra sin descargar nada.
# TODO(estudiante) 27a: cámbialo por tu partición real cuando `make data` funcione.
ruta = Path("tests/fixtures/muestra-valida.csv")
df = pd.read_csv(ruta, parse_dates=[fc.COL_TIEMPO])

print(df.shape)
print(loaders.leer_metadata())   # procedencia y hash, si ya descargaste datos
dc.resumen_contrato()

## 2. Estructura, tipos y nulos

Lo que hay que responder aquí, y que va directo a `docs/dataset-card.md`:

- ¿qué columnas hay y qué significa cada una (con **unidades**)?
- ¿dónde hay nulos y **por qué** los hay? Un nulo estructural (el campo no aplica)
  no se trata igual que un nulo por fallo de captura.
- ¿hay duplicados? ¿duplicados exactos o duplicados de clave?

In [ ]:
resumen = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "pct_nulos": (df.isna().mean() * 100).round(2),
        "unicos": df.nunique(),
    }
)
resumen

## 3. El eje temporal

Este es el punto que decide si el proyecto puede cumplir la fase de monitoreo.

- ¿el eje temporal es continuo o tiene huecos?
- ¿el volumen por período es estable, o hay meses con una fracción de los datos?
- ¿dónde conviene cortar referencia vs producción, y por qué **ahí**?

In [ ]:
por_periodo = (
    df.set_index(fc.COL_TIEMPO)
    .resample("D")
    .size()
    .rename("registros")
    .to_frame()
)
print(por_periodo.describe())
por_periodo.plot(title="Volumen por día", figsize=(11, 3));

## 4. Target y baseline honesto

- distribución del target: ¿asimétrica? ¿con outliers? ¿acotada?
- ¿cuál es el número que hay que batir? Predecir la mediana (o la clase mayoritaria)
  es el baseline. Si el modelo no le gana por un margen que le importa al negocio,
  el problema no está en los hiperparámetros.
- **split temporal**, nunca `train_test_split(shuffle=True)`: mezclar el futuro
  dentro del entrenamiento produce una métrica optimista que no se sostiene.

In [ ]:
limpio = fc.construir_features(loaders.limpiar(df))
train, test = loaders.split_temporal(limpio.sort_values(fc.COL_TIEMPO).reset_index(drop=True))
print(f"train: {len(train)} filas hasta {train[fc.COL_TIEMPO].max()}")
print(f"test:  {len(test)} filas desde {test[fc.COL_TIEMPO].min()}")

# TODO(estudiante) 27b: calcula aquí la métrica del baseline y anótala en el README.
train[fc.TARGET].describe()

## 5. Fugas de información (leakage)

Las tres que hay que descartar explícitamente, por escrito, antes de entrenar:

1. **escalar o imputar antes del split** — el estadístico usado se calcula con el
   test dentro;
2. **features construidas con información del futuro** — el `rolling` sin `shift(1)`
   es el caso clásico;
3. **el target codificado dentro de una feature** — un identificador que se asignó
   *después* de conocer el resultado.

**TODO(estudiante) 27c:** escribe una línea por cada una diciendo por qué tu
pipeline no la comete. Si no puedes descartarla, dilo: un riesgo declarado vale
más que un riesgo escondido.

In [ ]:
# Verificación barata pero efectiva: correlación del target con cada numérica.
# Una correlación de 0.99 casi siempre es leakage, no un modelo brillante.
numericas = [c for c in fc.FEATURES_NUMERICAS if c in limpio.columns]
limpio[[*numericas, fc.TARGET]].corr()[fc.TARGET].sort_values(ascending=False)